In [1]:
import pandas as pd
import pdfplumber
import os

data = pd.read_csv('../data/contributions-AIConsult2020.csv', encoding='latin-1', sep=';', on_bad_lines='skip')
pdf_col = 'You can upload a document here:\n\n'
pdf_dir = '../data/attachments-AIConsult2020/'

In [3]:

# List actual files in the PDF directory
actual_files = os.listdir('../data/attachments-AIConsult2020/')
print("ACTUAL FILES (sample):")
for f in actual_files[:10]:
    print(f)

print("\nCSV FILENAMES (sample):")
pdf_col = 'You can upload a document here:\n\n'
print(data[pdf_col].dropna().head(10).tolist())

ACTUAL FILES (sample):
F530387-ThinkTech_Statement_final.pdf
F529865-AI_Consultation_Summary_01.pdf
F530066-AI_public_consultation_Finnish_Consumer_Ombudsman_120620.pdf
F530105-FERMA_s_Position_Paper_on_Artificial_Intelligence.pdf
F528892-20200527_Vision_on_white_paper_AI.pdf
F529966-CONSULTATION_LIVRE_BLANC_EUROPEEN_29_Mai_consolid__avec_groupe_IA_CFDT_Cadres.pdf
F530373-CZ_nonpaper__EU_regulatory_framework_for_AI.pdf
F530016-Proposal_for_risk-based_approach.pdf
F514587-UE_Livre_blanc_sur_l_intelligence_artificielle___Une_approche_europ_enne_-_MASSET_J_r_me.pdf
F530130-European_Tech_Alliance_-_Addendum_Paper_to_EU_Public_Consultation_on_AI.pdf

CSV FILENAMES (sample):
['Governance_of_AI_Research_Group_EU_Commission_AI_White_Paper_Consultation.pdf', 'EIT_Health_Consultative_Group_on_EC_Data_Strategy_and_AI_White_Paper_-_31_May_2020_FINAL.pdf', 'feedback_Consultation_on_the_White_Paper_on_Artificial_Intelligence.docx', 'DIGITAL_SME_Position_Paper_AI_White_Paper_FINAL_DRAFT.pdf', 'CNMC_S

In [5]:
lookup = {}
for fname in actual_files:
    if '-' in fname:
        suffix = fname.split('-', 1)[1]  # everything after first dash
        lookup[suffix] = fname

def extract_pdf_text(csv_filename):
    if pd.isna(csv_filename):
        return None
    actual_fname = lookup.get(csv_filename)
    if not actual_fname:
        return None
    filepath = os.path.join('../data/attachments-AIConsult2020/', actual_fname)
    try:
        with pdfplumber.open(filepath) as pdf:
            text = "\n".join(page.extract_text() or '' for page in pdf.pages)
        return text if text.strip() else None
    except Exception as e:
        return None

In [6]:
sample = data[pdf_col].dropna().iloc[0]
text = extract_pdf_text(sample)
print(text[:500] if text else "No text extracted")

Ref. Ares(2020)3357682 - 26/06/2020
Contribution to a European Agenda for AI: Improving
Risk Management, Building Strong Governance,
Accelerating Education and Research
A Response to :
EU Commission White Paper
“On Artificial Intelligence -- A European approach to
excellence and trust”
Provided by:
The Governance of AI Research Group
Contributors include:
Adrien Abecassis, Harvard University
Justin B. Bullock, PhD, Texas A&M University
Johannes Himmelreich, PhD, Syracuse University
Valerie M. Hu


In [7]:
data['pdf_text'] = data[pdf_col].apply(extract_pdf_text)
print(f"Successfully extracted: {data['pdf_text'].notna().sum()}")
print(f"Failed/empty: {data['pdf_text'].isna().sum()}")

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Cannot set gray non-stroke color because /'p48' is an invalid float value


Successfully extracted: 369
Failed/empty: 847


In [9]:
no_pdf = data[pdf_col].isna().sum()
has_pdf_no_text = (data[pdf_col].notna() & data['pdf_text'].isna()).sum()
has_pdf_with_text = data['pdf_text'].notna().sum()

print(f"No PDF attached: {no_pdf}")
print(f"Had PDF but extraction failed: {has_pdf_no_text}")
print(f"Successfully extracted: {has_pdf_with_text}")

failed = data[data[pdf_col].notna() & data['pdf_text'].isna()]
print("\nFailed file extensions:")
print(failed[pdf_col].str.split('.').str[-1].value_counts())

No PDF attached: 793
Had PDF but extraction failed: 54
Successfully extracted: 369

Failed file extensions:
You can upload a document here:\n\n
docx    48
txt      2
doc      2
DOCX     1
PDF      1
Name: count, dtype: int64


In [10]:
data.to_pickle('../data/processed/data_with_pdf_text.pkl')
print("Saved!")

Saved!
